# Analyzing the Trends of COVID-19 with Python
**Intellipaat — Python for Data Science Capstone**

**Problem statement.** Given data about COVID-19 cases, visualize the impact and analyze the
**trend of infection and recovery**, then **forecast the number of cases expected about a week into the
future** based on current trends.

**Approach & tools (per the brief).** `pandas` for data wrangling, **Plotly** for interactive
visualizations, and **Facebook Prophet** for time-series forecasting.

**Contents**
1. Setup & data loading
2. Data cleaning & feature engineering
3. EDA — global infection & recovery trends
4. Geographic distribution (countries & WHO regions)
5. India deep-dive
6. Forecasting confirmed cases with Prophet (global, India, US, Brazil)
7. Conclusions

> **How to run:** keep `covid_19_clean_complete.csv` in the same folder, install
> `pip install -r requirements.txt`, then *Run All*. Plotly charts are **interactive** in
> Jupyter / Colab / nbviewer.

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import logging, warnings

from prophet import Prophet

warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
pio.templates.default = 'plotly_white'

# Brand colors
C = dict(Confirmed='#2563eb', Deaths='#dc2626', Recovered='#16a34a', Active='#f59e0b')

In [2]:
df = pd.read_csv('covid_19_clean_complete.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Province/State'] = df['Province/State'].fillna('')
df.head()

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,,Afghanistan,33.93911,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,,Albania,41.15330,20.168300,2020-01-22,0,0,0,0,Europe
2,,Algeria,28.03390,1.659600,2020-01-22,0,0,0,0,Africa
3,,Andorra,42.50630,1.521800,2020-01-22,0,0,0,0,Europe
4,,Angola,-11.20270,17.873900,2020-01-22,0,0,0,0,Africa


## 2. Data Understanding, Cleaning & Feature Engineering

In [3]:
print('Shape          :', df.shape)
print('Countries      :', df['Country/Region'].nunique())
print('Date range     :', df['Date'].min().date(), '->', df['Date'].max().date(),
      '(', df['Date'].nunique(), 'days )')
print('WHO regions    :', list(df['WHO Region'].unique()))
print('\nMissing values per column:')
print(df.isnull().sum())

Shape          : (49068, 10)
Countries      : 187
Date range     : 2020-01-22 -> 2020-07-27 ( 188 days )
WHO regions    : ['Eastern Mediterranean', 'Europe', 'Africa', 'Americas', 'Western Pacific', 'South-East Asia']

Missing values per column:
Province/State    0
Country/Region    0
Lat               0
Long              0
Date              0
Confirmed         0
Deaths            0
Recovered         0
Active            0
WHO Region        0
dtype: int64


The four numeric measures (`Confirmed`, `Deaths`, `Recovered`, `Active`) are **fully populated**;
only `Province/State` is sparse, which is expected — many countries report at the national level.
The data spans **187 countries** over **188 daily snapshots** (22 Jan – 27 Jul 2020). Records are
stored per province/country/day, so we **aggregate to a daily series** and derive the metrics we need.

In [4]:
# Global daily series (sum across all locations)
g = df.groupby('Date')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().sort_index()

# Daily NEW cases from the cumulative columns
for c in ['Confirmed', 'Deaths', 'Recovered']:
    g['New_' + c] = g[c].diff().fillna(g[c]).clip(lower=0)

# Rates (as % of confirmed)
g['Mortality_%'] = g['Deaths'] / g['Confirmed'] * 100
g['Recovery_%']  = g['Recovered'] / g['Confirmed'] * 100
g.tail()

,Confirmed,Deaths,Recovered,Active,New_Confirmed,New_Deaths,New_Recovered,Mortality_%,Recovery_%
Date,,,,,,,,,
2020-07-23,15510481,633506,8710969,6166006,282756.0,9966.0,169714.0,4.084374,56.161824
2020-07-24,15791645,639650,8939705,6212290,281164.0,6144.0,228736.0,4.050560,56.610347
2020-07-25,16047190,644517,9158743,6243930,255545.0,4867.0,219038.0,4.016385,57.073812
2020-07-26,16251796,648621,9293464,6309711,204606.0,4104.0,134721.0,3.991073,57.184228
2020-07-27,16480485,654036,9468087,6358362,228689.0,5415.0,174623.0,3.968548,57.450293


## 3. EDA — Global Infection & Recovery Trends

In [5]:
fig = go.Figure()
for c in ['Confirmed', 'Active', 'Recovered', 'Deaths']:
    fig.add_trace(go.Scatter(x=g.index, y=g[c], name=c, line=dict(color=C[c], width=2.5)))
fig.update_layout(title='Global COVID-19 Cumulative Cases Over Time',
                  yaxis_title='People', title_x=0.5, legend_title='')
fig

In [6]:
fig = go.Figure(go.Bar(x=g.index, y=g['New_Confirmed'], marker_color='#2563eb'))
fig.update_layout(title='Global Daily New Confirmed Cases', yaxis_title='New cases / day', title_x=0.5)
fig

Cumulative confirmed cases grow **exponentially** through the period, reaching **16.5M by 27 Jul 2020**.
Daily new cases climb relentlessly, **peaking at ~282,800 on 23 Jul 2020** — the pandemic was still
accelerating at the end of the data.

In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=g.index, y=g['Recovery_%'], name='Recovery rate %', line=dict(color='#16a34a', width=2.5)))
fig.add_trace(go.Scatter(x=g.index, y=g['Mortality_%'], name='Mortality rate %', line=dict(color='#dc2626', width=2.5)))
fig.update_layout(title='Global Recovery & Mortality Rate Over Time', yaxis_title='% of confirmed', title_x=0.5)
fig

The **recovery rate** climbs steadily to **~57%**, while the **mortality rate** spiked to ~7% early on
(when testing lagged) and then settled to **~4%** as testing and treatment improved. The widening gap
between recoveries and deaths is the clearest sign of improving outcomes over time.

## 4. Geographic Distribution

In [8]:
latest = df['Date'].max()
country_latest = df[df['Date'] == latest].groupby('Country/Region')[['Confirmed', 'Deaths']].sum()
top10 = country_latest.sort_values('Confirmed').tail(10)

fig = go.Figure(go.Bar(x=top10['Confirmed'], y=top10.index, orientation='h', marker_color='#2563eb',
                       text=[f'{v/1e6:.2f}M' for v in top10['Confirmed']], textposition='auto'))
fig.update_layout(title=f'Top 10 Countries by Confirmed Cases ({latest.date()})',
                  xaxis_title='Confirmed', title_x=0.5)
fig

In [9]:
# Interactive world map (renders in Jupyter / Colab / nbviewer)
name_fix = {'US': 'United States', 'Korea, South': 'South Korea', 'Taiwan*': 'Taiwan', 'Burma': 'Myanmar',
            'Congo (Kinshasa)': 'Democratic Republic of the Congo',
            'Congo (Brazzaville)': 'Republic of the Congo', "Cote d'Ivoire": 'Ivory Coast'}
cc = df[df['Date'] == latest].groupby('Country/Region')['Confirmed'].sum().reset_index()
cc['loc'] = cc['Country/Region'].replace(name_fix)
fig = px.choropleth(cc, locations='loc', locationmode='country names', color='Confirmed',
                    color_continuous_scale='Reds', title=f'Global Confirmed Cases by Country ({latest.date()})')
fig.update_layout(title_x=0.5)
fig

In [10]:
who = df.groupby(['Date', 'WHO Region'])['Confirmed'].sum().reset_index()
fig = px.area(who, x='Date', y='Confirmed', color='WHO Region', title='Confirmed Cases by WHO Region Over Time')
fig.update_layout(title_x=0.5)
fig

In [11]:
top6 = country_latest.sort_values('Confirmed', ascending=False).head(6).index.tolist()
fig = go.Figure()
pal = px.colors.qualitative.Bold
for i, ct in enumerate(top6):
    s = df[df['Country/Region'] == ct].groupby('Date')['Confirmed'].sum()
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=ct, line=dict(width=2.5, color=pal[i % len(pal)])))
fig.update_layout(title='Confirmed Cases: Top 6 Countries', yaxis_title='Confirmed', title_x=0.5)
fig

The **Americas** dominate the case load (US + Brazil alone exceed 6.7M), followed by **Europe** and
**South-East Asia** (driven by India). Case-fatality rates vary widely by country — e.g. the UK (~15%)
and Mexico (~11%) were far higher than India (~2.3%), reflecting differences in testing coverage,
demographics and health-system load.

## 5. India Deep-Dive

In [12]:
india = df[df['Country/Region'] == 'India'].groupby('Date')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().sort_index()
india['New_Confirmed'] = india['Confirmed'].diff().fillna(india['Confirmed']).clip(lower=0)

fig = go.Figure()
for c in ['Confirmed', 'Active', 'Recovered', 'Deaths']:
    fig.add_trace(go.Scatter(x=india.index, y=india[c], name=c, line=dict(color=C[c], width=2.5)))
fig.update_layout(title='India COVID-19 Cumulative Cases Over Time', yaxis_title='People', title_x=0.5)
fig

In [13]:
fig = go.Figure(go.Bar(x=india.index, y=india['New_Confirmed'], marker_color='#f59e0b'))
fig.update_layout(title='India Daily New Confirmed Cases', yaxis_title='New cases / day', title_x=0.5)
fig

India recorded its **first case on 30 Jan 2020**, stayed flat through the spring, then entered
**steep exponential growth** from June. By 27 Jul 2020 it had **1.48M confirmed cases** with a
**recovery rate of ~64%** (above the global average) and a relatively low **mortality rate of ~2.3%**.
Daily new cases were still rising sharply — **peaking near 50,000/day** at the end of the data.

## 6. Forecasting Confirmed Cases with Prophet

We model **cumulative confirmed cases** as a time series and use **Facebook Prophet** (piecewise-linear
trend + weekly seasonality) to forecast **30 days ahead**, with the **7-day-ahead** value highlighted to
match the brief. Each model is validated by holding out the **last 7 days** and measuring **MAPE**.

In [14]:
def country_series(country=None):
    d = df if country is None else df[df['Country/Region'] == country]
    return d.groupby('Date')['Confirmed'].sum().sort_index()

def fit_forecast(s, periods=30):
    data = pd.DataFrame({'ds': s.index, 'y': s.values})
    m = Prophet(weekly_seasonality=True, yearly_seasonality=False, daily_seasonality=False,
                changepoint_prior_scale=0.5, interval_width=0.95)
    m.fit(data)
    forecast = m.predict(m.make_future_dataframe(periods=periods))
    return forecast

def mape(actual, pred):
    actual, pred = np.array(actual, float), np.array(pred, float)
    return np.mean(np.abs((actual - pred) / actual)) * 100

def backtest_7day(s):
    fc = fit_forecast(s.iloc[:-7], periods=7)
    return mape(s.iloc[-7:].values, fc['yhat'].iloc[-7:].values)

In [15]:
# Fit + validate for the world and the three largest hotspots
targets = [('Global', None), ('India', 'India'), ('US', 'US'), ('Brazil', 'Brazil')]
rows, forecasts = [], {}
for label, country in targets:
    s = country_series(country)
    fc = fit_forecast(s, periods=30)
    forecasts[label] = (s, fc)
    last_date, last_val = s.index.max(), s.iloc[-1]
    d7  = fc.loc[fc['ds'] == last_date + pd.Timedelta(days=7),  'yhat'].iloc[0]
    d30 = fc.loc[fc['ds'] == last_date + pd.Timedelta(days=30), 'yhat'].iloc[0]
    rows.append({'Region': label,
                 'Last actual': f'{last_val:,.0f}',
                 '+7 days': f'{d7:,.0f}',
                 '+30 days': f'{d30:,.0f}',
                 '7-day backtest MAPE': f'{backtest_7day(s):.2f}%'})
summary = pd.DataFrame(rows)
summary

17:13:48 - cmdstanpy - INFO - Chain [1] start processing


17:13:48 - cmdstanpy - INFO - Chain [1] done processing


17:13:48 - cmdstanpy - INFO - Chain [1] start processing


17:13:48 - cmdstanpy - INFO - Chain [1] done processing


17:13:48 - cmdstanpy - INFO - Chain [1] start processing


...

17:13:48 - cmdstanpy - INFO - Chain [1] start processing


17:13:48 - cmdstanpy - INFO - Chain [1] done processing


17:13:48 - cmdstanpy - INFO - Chain [1] start processing


17:13:48 - cmdstanpy - INFO - Chain [1] done processing


,Region,Last actual,+7 days,+30 days,7-day backtest MAPE
0,Global,"16,480,485","17,598,852","22,328,258",3.76%
1,India,"1,480,073","1,531,948","2,145,817",14.29%
2,US,"4,290,259","4,582,593","5,853,834",5.65%
3,Brazil,"2,442,375","2,680,543","3,542,283",1.93%


In [16]:
def forecast_plot(label, color='#dc2626'):
    s, fc = forecasts[label]
    last = s.index.max(); fut = fc[fc['ds'] > last]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name='Observed', line=dict(color='#2563eb', width=2.5)))
    fig.add_trace(go.Scatter(x=fut['ds'], y=fut['yhat_upper'], line=dict(width=0), showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=fut['ds'], y=fut['yhat_lower'], fill='tonexty', fillcolor='rgba(220,38,38,0.15)',
                             line=dict(width=0), name='95% interval'))
    fig.add_trace(go.Scatter(x=fut['ds'], y=fut['yhat'], name='Forecast', line=dict(color=color, width=2.5, dash='dash')))
    fig.add_vline(x=last, line_dash='dot', line_color='gray')
    fig.update_layout(title=f'{label}: Confirmed Cases + 30-Day Prophet Forecast',
                      yaxis_title='Confirmed (cumulative)', title_x=0.5)
    return fig

forecast_plot('Global')

In [17]:
forecast_plot('India', color='#16a34a')

In [18]:
# Save all four forecasts (with 95% bands) to CSV
for label, (s, fc) in forecasts.items():
    out = fc[fc['ds'] > s.index.max()][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    out.columns = ['Date', 'Forecast', 'Lower_95', 'Upper_95']
    out[['Forecast', 'Lower_95', 'Upper_95']] = out[['Forecast', 'Lower_95', 'Upper_95']].round(0)
    out.to_csv(f'covid_{label.lower()}_forecast.csv', index=False)
print('Saved forecasts:', ', '.join(f'covid_{l.lower()}_forecast.csv' for l, _ in targets))

Saved forecasts: covid_global_forecast.csv, covid_india_forecast.csv, covid_us_forecast.csv, covid_brazil_forecast.csv


**Forecast results.** The global model is accurate (**~3.8% MAPE** at the 1-week horizon) and projects
confirmed cases rising from 16.5M to **~17.6M within a week** and **~22.3M within a month** if the trend
held. Brazil (1.9%) and the US (5.7%) also validate well. **India is the hardest to forecast (~14% MAPE)**
precisely because it was in its steepest acceleration phase — short-term linear extrapolation lags a curve
that is still bending upward, so the model slightly **under**-predicts India's near-term cases.

## 7. Conclusions

- **Infection trend:** Global cases grew exponentially to **16.5M** by 27 Jul 2020, with daily new cases
  still peaking (~283k/day) — no plateau yet.
- **Recovery vs mortality:** Recovery rate rose to **~57%** globally; mortality settled around **~4%** after
  an early testing-driven spike.
- **Geography:** The **Americas** led the case load (US, Brazil); **India** drove South-East Asia with fast
  late-stage growth but a comparatively low fatality rate (~2.3%).
- **Forecasting:** Prophet forecasts the next 7–30 days well for stable trajectories (Global ~3.8%,
  Brazil ~1.9%, US ~5.7% MAPE); rapidly accelerating curves like India's are inherently harder (~14%).

**Tools used:** pandas (wrangling) · Plotly (interactive visualization) · Facebook Prophet (forecasting) —
exactly as specified in the brief. Forecasts are exported to `covid_<region>_forecast.csv`.